# Frequent Itemset Analysis
This should get to the heart of the project: based on a list of cities I like, predict what other cities I might like.

I am going to artificially construct transactions by taking combinations of size n

First, I'll generate all the clusters, and create transactions from them.

In [1]:
import generate_frequent_itemsets as gfi

In [2]:
kmeans_clusters = gfi.generate_kmeans_clusters()

gmm_clusters = gfi.generate_gmm_clusters()

clusters = kmeans_clusters + gmm_clusters

In [3]:
# Generate frequent itemsets of size n from each cluster
transactions = []
for cluster in clusters:
    cluster_size = min(5, len(cluster)) #len(cluster) - 5 if len(cluster) > 5 else len(cluster)
    transactions.extend(gfi.generate_frequent_itemsets(cluster, n=cluster_size))

print(f"Total transactions generated: {len(transactions)}")
fr_transactions = [frozenset(t) for t in transactions]

# print first 5 transactions
for i in range(5):
    print(f"Transaction {i}: {transactions[i]}")

Total transactions generated: 5870303
Transaction 0: ['Albany', 'Albuquerque', 'Allentown', 'Anaheim', 'Ann_Arbor']
Transaction 1: ['Albany', 'Albuquerque', 'Allentown', 'Anaheim', 'Atlantic_City']
Transaction 2: ['Albany', 'Albuquerque', 'Allentown', 'Anaheim', 'Austin']
Transaction 3: ['Albany', 'Albuquerque', 'Allentown', 'Anaheim', 'Baltimore']
Transaction 4: ['Albany', 'Albuquerque', 'Allentown', 'Anaheim', 'Bend']


Now that we have almost 6 million 'transactions' (which are hopefully groups of similar cities), we can calculate some frequent itemset rules

In [16]:
from itertools import combinations

MIN_SUPPORT = 0.001 * len(transactions)  # tune this based on number of frequent itemsets. I guess I want a few hundred

def compute_support(candidates, transactions):
    """
    Return {itemset: count} keeping only itemsets with count >= MIN_SUPPORT.
    candidates: list of frozensets
    transactions: list of frozensets
    """
    to_return = {}
    for c in candidates:
        count = 0
        for t in transactions:
            if c.issubset(t):
                count += 1
        if count >= MIN_SUPPORT:
            to_return[c] = count
    return to_return

def optimized_compute_support(candidates, transactions):
    """
    Return {itemset: count} keeping only itemsets with count >= MIN_SUPPORT.
    Iterates transactions once: for each transaction, enumerate all k-subsets
    and increment counts only for those in the candidate set.
    O(T * C(5,k)) instead of O(|candidates| * T).
    candidates: list of frozensets (all must be the same size k)
    transactions: list of frozensets
    """
    candidate_set = set(candidates)
    counts = {c: 0 for c in candidates}
    k = len(next(iter(candidate_set))) if candidate_set else 0
    for t in transactions:
        for subset in combinations(t, k):
            fs = frozenset(subset)
            if fs in candidate_set:
                counts[fs] += 1
    return {c: count for c, count in counts.items() if count >= MIN_SUPPORT}

def apriori_gen(prev_frequent, k):
    """
    Generate candidate k-itemsets from frequent (k-1)-itemsets.
    Join step: combine two (k-1)-itemsets sharing their first k-2 items.
    Prune step: discard candidates whose (k-1)-subsets are not all frequent.
    prev_frequent: list of frozensets
    k: size of candidates to generate
    Returns a set of frozenset candidates.
    """
    prev_sorted = sorted([sorted(fs) for fs in prev_frequent])
    prev_set = set(prev_frequent)
    to_return = set()
    for i in range(len(prev_sorted)):
        for j in range(i+1, len(prev_sorted)):
            a, b = prev_sorted[i], prev_sorted[j]
            if a[:k-2] == b[:k-2]:
                candidate = frozenset(a) | frozenset(b)
                if len(candidate) < k:
                    continue
                if all(frozenset(comb) in prev_set for comb in combinations(candidate, k-1)):
                    to_return.add(candidate)
            else:
                break  # list is sorted — no further j can share this prefix
    return to_return


In [17]:
from collections import Counter

# 1-itemsets: single pass over transactions (much faster than compute_support for this step)
item_counts = Counter()
for t in fr_transactions:
    for item in t:
        item_counts[item] += 1

current_frequent = {frozenset([item]): count
                    for item, count in item_counts.items()
                    if count >= MIN_SUPPORT}
all_frequent = dict(current_frequent)
print(f"Frequent 1-itemsets: {len(current_frequent)}")

# Iteratively generate larger itemsets until no new frequent itemsets are found
k = 2
while current_frequent:
    candidates = list(apriori_gen(list(current_frequent.keys()), k))
    if not candidates:
        break
    current_frequent = optimized_compute_support(candidates, fr_transactions)
    all_frequent.update(current_frequent)
    print(f"Frequent {k}-itemsets: {len(current_frequent)}")
    k += 1

print(f"\nTotal frequent itemsets: {len(all_frequent)}")


Frequent 1-itemsets: 78
Frequent 2-itemsets: 1481
Frequent 3-itemsets: 0

Total frequent itemsets: 1559


Now I want to find the strongest 2-itemset association rules.

In [ ]:
n_transactions = len(fr_transactions)

def confidence(antecedent, consequent, frequent_itemsets):
    """P(consequent | antecedent) = support(A U C) / support(A)"""
    union = antecedent | consequent
    if union not in frequent_itemsets or antecedent not in frequent_itemsets:
        return 0.0
    return frequent_itemsets[union] / frequent_itemsets[antecedent]

# turns out this is actually not super helpful because of the way the 'transactions' were generated from clusters
def lift(antecedent, consequent, frequent_itemsets):
    """confidence(A → C) / support(C) — values > 1 indicate positive correlation"""
    union = antecedent | consequent
    if union not in frequent_itemsets or antecedent not in frequent_itemsets or consequent not in frequent_itemsets:
        return 0.0
    support_union = frequent_itemsets[union] / n_transactions
    support_ant   = frequent_itemsets[antecedent] / n_transactions
    support_con   = frequent_itemsets[consequent] / n_transactions
    return support_union / (support_ant * support_con)

In [29]:
MIN_CONFIDENCE = 0.081
MIN_LIFT = 0.0  # lift < 1 is expected here: each city's solo support is inflated by appearing
                # in many clusters, while pair support only comes from shared clusters

def generate_rules(frequent_itemsets, min_confidence=MIN_CONFIDENCE, min_lift=MIN_LIFT):
    """
    For every frequent itemset of size >= 2, generate all antecedent → consequent
    splits where consequent is a single item. Filters by confidence and lift.
    Returns a list of (antecedent, consequent, conf, lft) tuples, sorted by confidence desc.
    """
    rules = []
    for itemset in frequent_itemsets:
        if len(itemset) < 2:
            continue
        for item in itemset:
            consequent = frozenset([item])
            antecedent = itemset - consequent
            conf = confidence(antecedent, consequent, frequent_itemsets)
            lft  = lift(antecedent, consequent, frequent_itemsets)
            if conf >= min_confidence and lft >= min_lift:
                rules.append((antecedent, consequent, conf, lft))
    rules.sort(key=lambda r: r[2], reverse=True)
    return rules

rules = generate_rules(all_frequent)
print("Association rules generated: ", len(rules))


Association rules generated:  384


Now I need to write these association rules to a file to be looked up later.

In [ ]:
with open("assn_rules.txt", "w") as f:
    for ant, con, conf, lft in rules:
        ant_str = ", ".join(sorted(ant))
        con_str = ", ".join(sorted(con))
        f.write(f"{ant_str} => {con_str}\n")

print(f"Wrote {len(rules)} rules to assn_rules.txt")